# 03.4 LSTM Autoencoder (revised)

**Вход (артефакты из 03.1):**
- `split_metadata.json`, `clip_bounds.json`, `scaler.joblib`, `contract.json`

**Выход:**
- `lstm_main.pt`, `lstm_alt.pt` — best-by-val_loss модели (PyTorch state_dict)
- `lstm_config.json` — архитектура + параметры
- `lstm_training_history_main.json`, `lstm_training_history_alt.json` — losses per epoch
- `lstm_scores.parquet` — point-level scores (calval + test, with timestamp)
- `lstm_stability.json` — Jaccard / Pearson between main и alt

## Изменения vs первая версия 03.4

После критического review:

1. **Separate clean-calibration aggregators** — Pass C scoring теперь
   поддерживает дополнительные aggregator arrays, обновляемые ТОЛЬКО для
   окон, где split=='calibration_val' AND is_clean (window-level).
   Calibration ECDF строится по этим clean-window aggregators, не по
   point-level filter on общих aggregators. Это устраняет subtle leakage,
   где clean point получал score из окна с DQ-tainted соседями.
2. **Row_id contiguity assert** — extract_windows проверяет, что внутри
   окна row_ids = [start, start+1, ..., start+59] перед extraction.
   Catches silent mapping bug если annotated_v3 sorted не как ожидаем.
3. **`dq_soft_flag` в top_tainted preview** — consistency with IF/HDBSCAN.
4. **`timestamp` в lstm_scores.parquet** — consistency with IF/HDBSCAN.
5. **Assert non-empty calibration references** — defensive coding.
6. **`torch.amp` API** — future-proof PyTorch 2.0+.
7. **`np.savez` без compression** — Pass A/score I/O speedup.

## Архитектура

```
encoder:  Input(60, 12) → LSTM(12→64, 2 layers, dropout 0.1) → take final h → Linear(64→16) = z
decoder:  Linear(16→64) → repeat 60 times → LSTM(64→64, 2 layers) → Linear(64→12)
loss:     MSE per timestep per feature
```

Простая v2-style. Improvements от data policy.

## Архитектура passes

```
03.4.1 Build windows with carryover (train pool: stride=30, score pool: stride=15)
       + row_id contiguity assert
03.4.2 Phase-balanced sampling → train + val sets
03.4.3 Train LSTM-AE × 2 runs (seeds 1321, 2321), early stopping, AMP
03.4.4 Score calval+test windows → per-timestep errors
       Two pairs of aggregators:
         - general (all score-eligible windows)
         - clean-calibration (only calval + clean window)
03.4.5 Build lstm_scores.parquet via 2 passes (ECDF from clean-cal aggregators)
03.4.6 Stability (main vs alt)
03.4.7 Top-10 previews + summary
```

## 1. Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import os
import gc
import json
import time
import glob
from collections import defaultdict
import joblib

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
# Use new torch.amp API (works with PyTorch 2.0+)
try:
    from torch.amp import autocast, GradScaler
    NEW_AMP_API = True
except ImportError:
    from torch.cuda.amp import autocast, GradScaler
    NEW_AMP_API = False

DATA_DIR = '/content/drive/MyDrive/thesis_processed'
ARTIFACTS_DIR = os.path.join(DATA_DIR, 'models_v3_artifacts')
INPUT_PATH = os.path.join(DATA_DIR, 'european_flights_annotated_v3.parquet')

# 03.1 artifacts
SPLIT_PATH = os.path.join(ARTIFACTS_DIR, 'split_metadata.json')
CLIP_BOUNDS_PATH = os.path.join(ARTIFACTS_DIR, 'clip_bounds.json')
SCALER_PATH = os.path.join(ARTIFACTS_DIR, 'scaler.joblib')
CONTRACT_PATH = os.path.join(ARTIFACTS_DIR, 'contract.json')

# 03.4 outputs
LSTM_MAIN_PATH = os.path.join(ARTIFACTS_DIR, 'lstm_main.pt')
LSTM_ALT_PATH = os.path.join(ARTIFACTS_DIR, 'lstm_alt.pt')
LSTM_CONFIG_PATH = os.path.join(ARTIFACTS_DIR, 'lstm_config.json')
LSTM_HIST_MAIN_PATH = os.path.join(ARTIFACTS_DIR, 'lstm_training_history_main.json')
LSTM_HIST_ALT_PATH = os.path.join(ARTIFACTS_DIR, 'lstm_training_history_alt.json')
LSTM_SCORES_PATH = os.path.join(ARTIFACTS_DIR, 'lstm_scores.parquet')
LSTM_STABILITY_PATH = os.path.join(ARTIFACTS_DIR, 'lstm_stability.json')

# Temp dirs
LSTM_TRAIN_WINDOWS_DIR = os.path.join(ARTIFACTS_DIR, 'lstm_train_windows_temp')
LSTM_SCORE_WINDOWS_DIR = os.path.join(ARTIFACTS_DIR, 'lstm_score_windows_temp')

print(f'Input: {INPUT_PATH}')
print(f'Artifacts: {ARTIFACTS_DIR}')
for path in [INPUT_PATH, SPLIT_PATH, CLIP_BOUNDS_PATH, SCALER_PATH, CONTRACT_PATH]:
    print(f'  {os.path.basename(path):>22s}: {os.path.exists(path)}')

# GPU check
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'\nDevice: {device}')
if device.type == 'cuda':
    print(f'  GPU: {torch.cuda.get_device_name(0)}')
    print(f'  Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    print(f'  AMP API: {"new (torch.amp)" if NEW_AMP_API else "old (torch.cuda.amp)"}')

Mounted at /content/drive
Input: /content/drive/MyDrive/thesis_processed/european_flights_annotated_v3.parquet
Artifacts: /content/drive/MyDrive/thesis_processed/models_v3_artifacts
  european_flights_annotated_v3.parquet: True
     split_metadata.json: True
        clip_bounds.json: True
           scaler.joblib: True
           contract.json: True

Device: cuda
  GPU: NVIDIA A100-SXM4-80GB
  Memory: 85.1 GB
  AMP API: new (torch.amp)


## 2. Load 03.1 artifacts + configuration

In [2]:
with open(CONTRACT_PATH) as f:
    contract = json.load(f)

MODEL_FEATURES = contract['model_features']
DQ_HARD = contract['dq_hard_flag']
DQ_SOFT = contract['dq_soft_flag']
FEATURE_QUALITY = contract['feature_quality_flag']
GAP_FLAG = contract['gap_flag']
LSTM_DT_THRESHOLD = contract['lstm_dt_threshold_sec']  # 2.0
RANDOM_STATE = contract['random_state']

with open(SPLIT_PATH) as f:
    split_metadata = json.load(f)

train_core_ids = np.array(sorted(split_metadata['train_core_flights']), dtype=np.int64)
calibration_val_ids = np.array(sorted(split_metadata['calibration_val_flights']), dtype=np.int64)
test_ids = np.array(sorted(split_metadata['test_flights']), dtype=np.int64)

with open(CLIP_BOUNDS_PATH) as f:
    clip_data = json.load(f)
clip_bounds = clip_data['bounds']

scaler = joblib.load(SCALER_PATH)

# === Configuration ===
N_FEATURES = 12
SEQ_LEN = 60
TRAIN_STRIDE = 30
SCORE_STRIDE = 15

HIDDEN_SIZE = 64
LATENT_DIM = 16
NUM_LAYERS = 2
DROPOUT = 0.1

BATCH_SIZE_TRAIN = 256
BATCH_SIZE_SCORE = 512
LEARNING_RATE = 1e-3
MAX_EPOCHS = 40
PATIENCE = 5
MIN_DELTA = 1e-4
GRAD_CLIP = 1.0
USE_AMP = device.type == 'cuda'

TRAIN_WINDOWS_PER_GROUP = 150_000
VAL_WINDOWS_PER_GROUP = 30_000

LSTM_VAL_FRACTION = 0.20

RUN_SEEDS = {'main': RANDOM_STATE, 'alt': RANDOM_STATE + 1000}  # 1321, 2321

PHASE_NAMES = {-1: 'unknown', 0: 'ground', 1: 'takeoff', 2: 'climb',
               3: 'cruise', 4: 'descent', 5: 'approach', 6: 'landing'}
PHASE_GROUP_MAP = {
    2: 'climb', 3: 'cruise', 4: 'descent',
    0: 'terminal', 1: 'terminal', 5: 'terminal', 6: 'terminal',
}
PHASE_GROUPS = ['climb', 'cruise', 'descent', 'terminal']

print(f'\nConfiguration:')
print(f'  seq_len:              {SEQ_LEN}')
print(f'  train_stride:         {TRAIN_STRIDE}')
print(f'  score_stride:         {SCORE_STRIDE}')
print(f'  lstm_dt_threshold:    {LSTM_DT_THRESHOLD}s')
print(f'  hidden_size:          {HIDDEN_SIZE}')
print(f'  latent_dim:           {LATENT_DIM}')
print(f'  num_layers:           {NUM_LAYERS}')
print(f'  batch_size:           {BATCH_SIZE_TRAIN} (train) / {BATCH_SIZE_SCORE} (score)')
print(f'  max_epochs:           {MAX_EPOCHS}')
print(f'  patience:             {PATIENCE}')
print(f'  mixed_precision:      {USE_AMP}')
print(f'  train_cap/group:      {TRAIN_WINDOWS_PER_GROUP:,}')
print(f'  val_cap/group:        {VAL_WINDOWS_PER_GROUP:,}')
print(f'  internal val frac:    {LSTM_VAL_FRACTION:.0%}')
print(f'  runs:                 {list(RUN_SEEDS.keys())}, seeds {list(RUN_SEEDS.values())}')


Configuration:
  seq_len:              60
  train_stride:         30
  score_stride:         15
  lstm_dt_threshold:    2.0s
  hidden_size:          64
  latent_dim:           16
  num_layers:           2
  batch_size:           256 (train) / 512 (score)
  max_epochs:           40
  patience:             5
  mixed_precision:      True
  train_cap/group:      150,000
  val_cap/group:        30,000
  internal val frac:    20%
  runs:                 ['main', 'alt'], seeds [1321, 2321]


## 3. Internal train_core split for LSTM val

In [3]:
rng = np.random.RandomState(RANDOM_STATE)
shuffled_train_core = rng.permutation(train_core_ids)
n_train_core = len(shuffled_train_core)
n_lstm_val = int(n_train_core * LSTM_VAL_FRACTION)
lstm_val_flights = shuffled_train_core[:n_lstm_val]
lstm_train_flights = shuffled_train_core[n_lstm_val:]

lstm_train_set = np.array(sorted(lstm_train_flights), dtype=np.int64)
lstm_val_set = np.array(sorted(lstm_val_flights), dtype=np.int64)

print(f'\nInternal LSTM split (within train_core):')
print(f'  lstm_train_flights: {len(lstm_train_set):,}')
print(f'  lstm_val_flights:   {len(lstm_val_set):,}')

assert len(set(lstm_train_set) & set(lstm_val_set)) == 0, 'lstm_train/val overlap!'
assert len(lstm_train_set) + len(lstm_val_set) == n_train_core, 'split coverage mismatch!'
print('  Splits mutually exclusive and exhaustive ✓')


Internal LSTM split (within train_core):
  lstm_train_flights: 13,784
  lstm_val_flights:   3,446
  Splits mutually exclusive and exhaustive ✓


## 4. Helper: clip + scale features

In [4]:
def clip_and_scale_features(df, model_features=MODEL_FEATURES,
                            clip_bounds=clip_bounds, scaler=scaler):
    X = df[model_features].to_numpy(dtype=np.float32, copy=True)
    for i, feature in enumerate(model_features):
        bounds = clip_bounds[feature]
        np.clip(X[:, i], bounds['low'], bounds['high'], out=X[:, i])
    X_scaled = scaler.transform(X).astype(np.float32)
    for i, feature in enumerate(model_features):
        df[feature] = X_scaled[:, i]
    return df


print('clip_and_scale_features defined.')

clip_and_scale_features defined.


## 5. Pass A — build windows with carryover (+ row_id contiguity assert)

Single streaming pass через annotated_v3 с carryover для целых рейсов.
Windows extracted within complete flights (после combine с carry).

Train pool: stride=30, strict eligibility (finite + no DQ + no dt_sec>2).
Score pool: stride=15, loose eligibility (finite + no dt_sec>2; DQ-tainted OK).

**Critical new assert:** проверяем row_id contiguity внутри окна — это
обязательная предпосылка для correctness window→point mapping. Если
row_ids не consecutive, mapping молча испортится.

`np.savez` (без compression) — speedup vs первая версия.

In [5]:
print('Pass A: building train/score window pools with carryover...')
t0 = time.time()

# Cleanup temp
os.makedirs(LSTM_TRAIN_WINDOWS_DIR, exist_ok=True)
os.makedirs(LSTM_SCORE_WINDOWS_DIR, exist_ok=True)
for tmp_dir in [LSTM_TRAIN_WINDOWS_DIR, LSTM_SCORE_WINDOWS_DIR]:
    old = glob.glob(os.path.join(tmp_dir, '*.npz')) + \
          glob.glob(os.path.join(tmp_dir, '*.parquet'))
    if old:
        for f in old:
            os.remove(f)
        print(f'Removed {len(old)} stale files from {os.path.basename(tmp_dir)}')

read_cols = (
    ['flight_id', 'timestamp', 'flight_phase',
     DQ_HARD, DQ_SOFT, FEATURE_QUALITY, GAP_FLAG]
    + MODEL_FEATURES
)


def extract_windows_from_flight_chunk(df_complete, stride):
    """Extract windows of seq_len=60 with given stride for given flights.

    Returns: (windows_arr, meta_df) или (None, None) if no eligible windows.

    Includes critical assert: row_ids within each window must be contiguous.
    """
    df_complete = df_complete.copy()
    df_complete['model_features_finite'] = (
        df_complete[MODEL_FEATURES].notna().all(axis=1).astype('uint8'))

    dt_arr = df_complete['dt_sec'].to_numpy() if 'dt_sec' in df_complete.columns else None
    if dt_arr is None:
        df_complete['timestamp'] = pd.to_datetime(df_complete['timestamp'])
        df_complete['dt_sec'] = df_complete.groupby('flight_id')['timestamp'].diff(
        ).dt.total_seconds().fillna(1.0)
        dt_arr = df_complete['dt_sec'].to_numpy()

    df_complete['dt_invalid'] = (dt_arr > LSTM_DT_THRESHOLD).astype('uint8')

    df_complete = df_complete.sort_values(['flight_id', 'timestamp']).reset_index(drop=True)

    windows_features = []
    meta_records = []

    for fid, fgrp in df_complete.groupby('flight_id', sort=False):
        if len(fgrp) < SEQ_LEN:
            continue

        if fid in lstm_train_set:
            f_split = 'lstm_train'
        elif fid in lstm_val_set:
            f_split = 'lstm_val'
        elif fid in calibration_val_ids:
            f_split = 'calibration_val'
        elif fid in test_ids:
            f_split = 'test'
        else:
            continue

        fgrp_arr = fgrp[MODEL_FEATURES].to_numpy(dtype=np.float32)
        row_ids = fgrp['row_id'].to_numpy()
        phases = fgrp['flight_phase'].to_numpy()

        finite_arr = fgrp['model_features_finite'].to_numpy()
        dq_hard_arr = fgrp[DQ_HARD].to_numpy()
        dq_soft_arr = fgrp[DQ_SOFT].to_numpy()
        fq_arr = fgrp[FEATURE_QUALITY].to_numpy()
        dt_invalid_arr = fgrp['dt_invalid'].to_numpy()

        n = len(fgrp)
        for start in range(0, n - SEQ_LEN + 1, stride):
            end = start + SEQ_LEN

            # === CRITICAL ASSERT: row_id contiguity ===
            w_row_ids = row_ids[start:end]
            if not np.array_equal(
                w_row_ids,
                np.arange(w_row_ids[0], w_row_ids[0] + SEQ_LEN, dtype=w_row_ids.dtype)
            ):
                raise ValueError(
                    f'Non-contiguous row_id in window: flight_id={fid}, '
                    f'start={start}, row_id_start={w_row_ids[0]}, '
                    f'row_id_end={w_row_ids[-1]}, expected_end='
                    f'{w_row_ids[0] + SEQ_LEN - 1}'
                )

            window_feat = fgrp_arr[start:end]
            if not np.isfinite(window_feat).all():
                continue

            w_finite = finite_arr[start:end].all()
            w_dq_hard = dq_hard_arr[start:end].any()
            w_dq_soft = dq_soft_arr[start:end].any()
            w_fq = fq_arr[start:end].any()
            w_dt_bad = dt_invalid_arr[start:end].any()

            is_train_elig = bool(
                w_finite and not w_dq_hard and not w_dq_soft
                and not w_fq and not w_dt_bad
            )
            is_score_elig = bool(w_finite and not w_dt_bad)
            is_clean = bool(not w_dq_hard and not w_dq_soft and not w_fq)

            w_phases = phases[start:end]
            uniq, counts = np.unique(w_phases, return_counts=True)
            dominant_phase = int(uniq[counts.argmax()])
            phase_group = PHASE_GROUP_MAP.get(dominant_phase, 'unknown')

            meta_records.append({
                'flight_id': int(fid),
                'start_row_id': int(row_ids[start]),
                'end_row_id': int(row_ids[end - 1]),
                'dominant_phase': dominant_phase,
                'phase_group': phase_group,
                'split': f_split,
                'is_train_eligible': is_train_elig,
                'is_score_eligible': is_score_elig,
                'is_clean': is_clean,
                'w_dq_hard': int(w_dq_hard),
                'w_dq_soft': int(w_dq_soft),
                'w_fq': int(w_fq),
            })
            windows_features.append(window_feat)

    if not windows_features:
        return None, None

    windows_arr = np.stack(windows_features, axis=0).astype(np.float32)
    meta_df = pd.DataFrame(meta_records)
    return windows_arr, meta_df


def process_complete_flights(df_complete, batch_idx):
    """Generate train pool (stride=30) and score pool (stride=15) chunks."""
    if len(df_complete) == 0:
        return 0, 0

    train_pool_fids = set(lstm_train_set.tolist()) | set(lstm_val_set.tolist())
    df_train_only = df_complete[df_complete['flight_id'].isin(train_pool_fids)].copy()
    n_train = 0
    if len(df_train_only) > 0:
        win_train, meta_train = extract_windows_from_flight_chunk(
            df_train_only, stride=TRAIN_STRIDE)
        if win_train is not None:
            train_mask = meta_train['is_train_eligible'].to_numpy()
            if train_mask.any():
                win_train_filt = win_train[train_mask]
                meta_train_filt = meta_train[train_mask].reset_index(drop=True)
                np.savez(
                    os.path.join(LSTM_TRAIN_WINDOWS_DIR, f'win_{batch_idx:03d}.npz'),
                    windows=win_train_filt
                )
                meta_train_filt.to_parquet(
                    os.path.join(LSTM_TRAIN_WINDOWS_DIR, f'meta_{batch_idx:03d}.parquet'),
                    index=False
                )
                n_train = len(win_train_filt)

    score_pool_fids = set(calibration_val_ids.tolist()) | set(test_ids.tolist())
    df_score_only = df_complete[df_complete['flight_id'].isin(score_pool_fids)].copy()
    n_score = 0
    if len(df_score_only) > 0:
        win_score, meta_score = extract_windows_from_flight_chunk(
            df_score_only, stride=SCORE_STRIDE)
        if win_score is not None:
            score_mask = meta_score['is_score_eligible'].to_numpy()
            if score_mask.any():
                win_score_filt = win_score[score_mask]
                meta_score_filt = meta_score[score_mask].reset_index(drop=True)
                np.savez(
                    os.path.join(LSTM_SCORE_WINDOWS_DIR, f'win_{batch_idx:03d}.npz'),
                    windows=win_score_filt
                )
                meta_score_filt.to_parquet(
                    os.path.join(LSTM_SCORE_WINDOWS_DIR, f'meta_{batch_idx:03d}.parquet'),
                    index=False
                )
                n_score = len(win_score_filt)

    return n_train, n_score


# Streaming pass with carryover
pf = pq.ParquetFile(INPUT_PATH)
read_cols_full = read_cols + ['dt_sec']

batch_size = 5_000_000
carry = pd.DataFrame()
global_offset = 0
n_batches = 0
n_train_total = 0
n_score_total = 0

for batch in pf.iter_batches(batch_size=batch_size, columns=read_cols_full):
    df_new = batch.to_pandas()
    n_new = len(df_new)
    n_batches += 1

    df_new['row_id'] = np.arange(global_offset, global_offset + n_new, dtype=np.int64)
    global_offset += n_new

    df_new = clip_and_scale_features(df_new)

    if len(carry) > 0:
        df_combined = pd.concat([carry, df_new], ignore_index=True)
    else:
        df_combined = df_new

    df_combined = df_combined.sort_values(['flight_id', 'timestamp']).reset_index(drop=True)

    last_fid = df_combined['flight_id'].iloc[-1]
    last_mask = df_combined['flight_id'] == last_fid
    process_df = df_combined[~last_mask].copy()
    carry = df_combined[last_mask].copy()

    if len(process_df) > 0:
        n_t, n_s = process_complete_flights(process_df, n_batches - 1)
        n_train_total += n_t
        n_score_total += n_s

    print(f'  batch {n_batches}: rows={n_new:,}, carry={len(carry):,}, '
          f'train_w={n_train_total:,}, score_w={n_score_total:,}', end='\r', flush=True)

    del df_new, df_combined, process_df
    gc.collect()

if len(carry) > 0:
    n_t, n_s = process_complete_flights(carry, n_batches)
    n_train_total += n_t
    n_score_total += n_s
    print(f'\n  Final carry: +train={n_t:,}, +score={n_s:,}')

del carry
gc.collect()

print()
print(f'\nPass A complete: {time.time() - t0:.0f}s')
print(f'Total train-eligible windows: {n_train_total:,}')
print(f'Total score-eligible windows: {n_score_total:,}')
print('Row_id contiguity check: PASSED (no assertion failures)')

Pass A: building train/score window pools with carryover...

  Final carry: +train=0, +score=566


Pass A complete: 833s
Total train-eligible windows: 2,638,580
Total score-eligible windows: 4,074,405
Row_id contiguity check: PASSED (no assertion failures)


## 6. Combine meta + phase-balanced sampling

In [6]:
print('Combining train meta chunks...')
t0 = time.time()

meta_chunks = sorted(glob.glob(os.path.join(LSTM_TRAIN_WINDOWS_DIR, 'meta_*.parquet')))
print(f'Found {len(meta_chunks)} train meta chunks')

all_train_meta = []
window_offset_cumsum = 0
chunk_offsets = {}

for cp in meta_chunks:
    chunk_idx = int(os.path.basename(cp).split('_')[1].split('.')[0])
    meta = pd.read_parquet(cp)
    meta['chunk_idx'] = chunk_idx
    meta['local_idx'] = np.arange(len(meta), dtype=np.int64)
    meta['global_window_id'] = np.arange(
        window_offset_cumsum, window_offset_cumsum + len(meta), dtype=np.int64)
    chunk_offsets[chunk_idx] = window_offset_cumsum
    window_offset_cumsum += len(meta)
    all_train_meta.append(meta)

train_meta = pd.concat(all_train_meta, ignore_index=True)
del all_train_meta
gc.collect()
print(f'Total train-eligible windows: {len(train_meta):,}')

print(f'\nBy split:')
for sp in ['lstm_train', 'lstm_val']:
    n = (train_meta['split'] == sp).sum()
    print(f'  {sp}: {n:,}')

print(f'\nBy phase_group (lstm_train flights):')
for grp in PHASE_GROUPS:
    n = ((train_meta['phase_group'] == grp) & (train_meta['split'] == 'lstm_train')).sum()
    print(f'  {grp:>10s}: {n:,}')

print(f'\nBy phase_group (lstm_val flights):')
for grp in PHASE_GROUPS:
    n = ((train_meta['phase_group'] == grp) & (train_meta['split'] == 'lstm_val')).sum()
    print(f'  {grp:>10s}: {n:,}')

print(f'\nPhase-balanced sampling: train cap={TRAIN_WINDOWS_PER_GROUP:,}/group, '
      f'val cap={VAL_WINDOWS_PER_GROUP:,}/group')

rng = np.random.RandomState(RANDOM_STATE)
train_selected_idx = []
val_selected_idx = []

for grp in PHASE_GROUPS:
    pool_train = train_meta[
        (train_meta['phase_group'] == grp) & (train_meta['split'] == 'lstm_train')
    ].index.to_numpy()
    n_pool_train = len(pool_train)
    n_take_train = min(TRAIN_WINDOWS_PER_GROUP, n_pool_train)
    if n_pool_train > 0:
        chosen = rng.choice(pool_train, size=n_take_train, replace=False)
        train_selected_idx.extend(chosen.tolist())
        print(f'  {grp:>10s} train: pool={n_pool_train:,}, take={n_take_train:,}')

    pool_val = train_meta[
        (train_meta['phase_group'] == grp) & (train_meta['split'] == 'lstm_val')
    ].index.to_numpy()
    n_pool_val = len(pool_val)
    n_take_val = min(VAL_WINDOWS_PER_GROUP, n_pool_val)
    if n_pool_val > 0:
        chosen_v = rng.choice(pool_val, size=n_take_val, replace=False)
        val_selected_idx.extend(chosen_v.tolist())
        print(f'  {grp:>10s} val:   pool={n_pool_val:,}, take={n_take_val:,}')

train_selected_idx = np.array(sorted(train_selected_idx), dtype=np.int64)
val_selected_idx = np.array(sorted(val_selected_idx), dtype=np.int64)

print(f'\nFinal training set: {len(train_selected_idx):,} windows')
print(f'Final val set:      {len(val_selected_idx):,} windows')

mem_train = len(train_selected_idx) * SEQ_LEN * N_FEATURES * 4 / 1e9
mem_val = len(val_selected_idx) * SEQ_LEN * N_FEATURES * 4 / 1e9
print(f'\nMemory estimate (RAM):')
print(f'  train: {mem_train:.2f} GB')
print(f'  val:   {mem_val:.2f} GB')

print(f'Sampling complete: {time.time() - t0:.0f}s')

Combining train meta chunks...
Found 22 train meta chunks
Total train-eligible windows: 2,638,580

By split:
  lstm_train: 2,113,159
  lstm_val: 525,421

By phase_group (lstm_train flights):
       climb: 396,763
      cruise: 1,114,793
     descent: 359,674
    terminal: 241,929

By phase_group (lstm_val flights):
       climb: 99,250
      cruise: 276,280
     descent: 89,432
    terminal: 60,459

Phase-balanced sampling: train cap=150,000/group, val cap=30,000/group
       climb train: pool=396,763, take=150,000
       climb val:   pool=99,250, take=30,000
      cruise train: pool=1,114,793, take=150,000
      cruise val:   pool=276,280, take=30,000
     descent train: pool=359,674, take=150,000
     descent val:   pool=89,432, take=30,000
    terminal train: pool=241,929, take=150,000
    terminal val:   pool=60,459, take=30,000

Final training set: 600,000 windows
Final val set:      120,000 windows

Memory estimate (RAM):
  train: 1.73 GB
  val:   0.35 GB
Sampling complete: 7s


## 7. Load sampled windows into RAM

In [7]:
print('Loading sampled train/val windows into RAM...')
t0 = time.time()


def load_windows_by_meta(meta_subset, windows_dir):
    grouped = meta_subset.groupby('chunk_idx')
    arrays = []
    for chunk_idx, grp in grouped:
        win_path = os.path.join(windows_dir, f'win_{int(chunk_idx):03d}.npz')
        chunk_windows = np.load(win_path)['windows']
        local_indices = grp['local_idx'].to_numpy()
        arrays.append(chunk_windows[local_indices])
    return np.concatenate(arrays, axis=0).astype(np.float32) if arrays else np.empty((0, SEQ_LEN, N_FEATURES), dtype=np.float32)


train_meta_subset = train_meta.iloc[train_selected_idx].sort_values(['chunk_idx', 'local_idx'])
X_train = load_windows_by_meta(train_meta_subset, LSTM_TRAIN_WINDOWS_DIR)
print(f'  X_train shape: {X_train.shape}, memory: {X_train.nbytes / 1e9:.2f} GB')

val_meta_subset = train_meta.iloc[val_selected_idx].sort_values(['chunk_idx', 'local_idx'])
X_val = load_windows_by_meta(val_meta_subset, LSTM_TRAIN_WINDOWS_DIR)
print(f'  X_val shape:   {X_val.shape}, memory: {X_val.nbytes / 1e9:.2f} GB')

del train_meta, train_meta_subset, val_meta_subset
gc.collect()
print(f'\nLoad complete: {time.time() - t0:.0f}s')

Loading sampled train/val windows into RAM...
  X_train shape: (600000, 60, 12), memory: 1.73 GB
  X_val shape:   (120000, 60, 12), memory: 0.35 GB

Load complete: 31s


## 8. LSTM Autoencoder model

In [8]:
class LSTMAutoencoder(nn.Module):
    def __init__(self, n_features=12, seq_len=60, hidden_size=64, latent_dim=16,
                 num_layers=2, dropout=0.1):
        super().__init__()
        self.seq_len = seq_len
        self.hidden_size = hidden_size

        self.encoder_lstm = nn.LSTM(
            input_size=n_features,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=True,
        )
        self.encoder_fc = nn.Linear(hidden_size, latent_dim)

        self.decoder_fc = nn.Linear(latent_dim, hidden_size)
        self.decoder_lstm = nn.LSTM(
            input_size=hidden_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=True,
        )
        self.output_fc = nn.Linear(hidden_size, n_features)

    def forward(self, x):
        _, (h_n, _) = self.encoder_lstm(x)
        h_last = h_n[-1]
        latent = self.encoder_fc(h_last)

        dec_input = self.decoder_fc(latent)
        dec_input_seq = dec_input.unsqueeze(1).repeat(1, self.seq_len, 1)

        dec_out, _ = self.decoder_lstm(dec_input_seq)
        out = self.output_fc(dec_out)
        return out


print('LSTMAutoencoder defined.')

LSTMAutoencoder defined.


## 9. Dataset

In [9]:
class WindowsDataset(Dataset):
    def __init__(self, X):
        self.X = torch.from_numpy(X)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx]


train_ds = WindowsDataset(X_train)
val_ds = WindowsDataset(X_val)
print(f'Train dataset: {len(train_ds):,}, Val dataset: {len(val_ds):,}')

Train dataset: 600,000, Val dataset: 120,000


## 10. Training function (with new torch.amp API)

In [10]:
def _autocast_ctx():
    """Return AMP context manager."""
    if not USE_AMP:
        return torch.no_grad if False else (lambda: __import__('contextlib').nullcontext())
    if NEW_AMP_API:
        return lambda: autocast(device_type='cuda')
    else:
        return lambda: autocast()


def train_lstm_run(seed, save_path, history_path):
    print(f'\n=== Training LSTM-AE (seed={seed}) ===')

    torch.manual_seed(seed)
    np.random.seed(seed)
    if device.type == 'cuda':
        torch.cuda.manual_seed_all(seed)

    model = LSTMAutoencoder(
        n_features=N_FEATURES, seq_len=SEQ_LEN, hidden_size=HIDDEN_SIZE,
        latent_dim=LATENT_DIM, num_layers=NUM_LAYERS, dropout=DROPOUT,
    ).to(device)

    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'  Model parameters: {n_params:,}')

    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    criterion = nn.MSELoss()
    # GradScaler init: new API requires device_type
    if USE_AMP:
        if NEW_AMP_API:
            amp_scaler = GradScaler('cuda')
        else:
            amp_scaler = GradScaler()
    else:
        amp_scaler = None

    train_loader = DataLoader(
        train_ds, batch_size=BATCH_SIZE_TRAIN, shuffle=True,
        num_workers=2, pin_memory=True, drop_last=True
    )
    val_loader = DataLoader(
        val_ds, batch_size=BATCH_SIZE_TRAIN, shuffle=False,
        num_workers=2, pin_memory=True
    )

    history = {'train_loss': [], 'val_loss': [], 'epoch_time': []}
    best_val_loss = float('inf')
    epochs_no_improve = 0
    best_epoch = -1

    for epoch in range(MAX_EPOCHS):
        t0_epoch = time.time()

        # === Train ===
        model.train()
        train_loss_sum = 0.0
        n_train_batches = 0
        for x in train_loader:
            x = x.to(device, non_blocking=True)
            optimizer.zero_grad()
            if USE_AMP:
                if NEW_AMP_API:
                    with autocast(device_type='cuda'):
                        recon = model(x)
                        loss = criterion(recon, x)
                else:
                    with autocast():
                        recon = model(x)
                        loss = criterion(recon, x)
                amp_scaler.scale(loss).backward()
                amp_scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                amp_scaler.step(optimizer)
                amp_scaler.update()
            else:
                recon = model(x)
                loss = criterion(recon, x)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                optimizer.step()
            train_loss_sum += loss.item()
            n_train_batches += 1

        avg_train_loss = train_loss_sum / max(n_train_batches, 1)

        # === Val ===
        model.eval()
        val_loss_sum = 0.0
        n_val_batches = 0
        with torch.no_grad():
            for x in val_loader:
                x = x.to(device, non_blocking=True)
                if USE_AMP:
                    if NEW_AMP_API:
                        with autocast(device_type='cuda'):
                            recon = model(x)
                            loss = criterion(recon, x)
                    else:
                        with autocast():
                            recon = model(x)
                            loss = criterion(recon, x)
                else:
                    recon = model(x)
                    loss = criterion(recon, x)
                val_loss_sum += loss.item()
                n_val_batches += 1
        avg_val_loss = val_loss_sum / max(n_val_batches, 1)

        epoch_time = time.time() - t0_epoch
        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(avg_val_loss)
        history['epoch_time'].append(epoch_time)

        improved = avg_val_loss < best_val_loss - MIN_DELTA
        if improved:
            best_val_loss = avg_val_loss
            best_epoch = epoch
            epochs_no_improve = 0
            torch.save(model.state_dict(), save_path)
            marker = ' ← BEST'
        else:
            epochs_no_improve += 1
            marker = ''

        print(f'  epoch {epoch + 1:>2d}/{MAX_EPOCHS}: '
              f'train_loss={avg_train_loss:.6f}, val_loss={avg_val_loss:.6f}, '
              f'time={epoch_time:.0f}s{marker}', flush=True)

        if epochs_no_improve >= PATIENCE:
            print(f'  Early stopping at epoch {epoch + 1}. Best val: {best_val_loss:.6f} '
                  f'(epoch {best_epoch + 1})')
            break

    history['best_val_loss'] = best_val_loss
    history['best_epoch'] = best_epoch + 1
    history['n_train_windows'] = len(X_train)
    history['n_val_windows'] = len(X_val)
    with open(history_path, 'w') as f:
        json.dump(history, f, indent=2)

    return best_val_loss, best_epoch + 1

## 11. Train 2 runs

In [11]:
t0 = time.time()
run_results = {}
for run_name, seed in RUN_SEEDS.items():
    save_path = LSTM_MAIN_PATH if run_name == 'main' else LSTM_ALT_PATH
    hist_path = LSTM_HIST_MAIN_PATH if run_name == 'main' else LSTM_HIST_ALT_PATH
    best_val, best_ep = train_lstm_run(seed, save_path, hist_path)
    run_results[run_name] = {'best_val_loss': best_val, 'best_epoch': best_ep}

print(f'\nTraining complete: {time.time() - t0:.0f}s')
for r, info in run_results.items():
    print(f'  {r}: best_val_loss={info["best_val_loss"]:.6f} @ epoch {info["best_epoch"]}')

del X_train, X_val, train_ds, val_ds
gc.collect()
if device.type == 'cuda':
    torch.cuda.empty_cache()

# Save config
lstm_config = {
    'n_features': N_FEATURES,
    'seq_len': SEQ_LEN,
    'hidden_size': HIDDEN_SIZE,
    'latent_dim': LATENT_DIM,
    'num_layers': NUM_LAYERS,
    'dropout': DROPOUT,
    'batch_size_train': BATCH_SIZE_TRAIN,
    'batch_size_score': BATCH_SIZE_SCORE,
    'learning_rate': LEARNING_RATE,
    'max_epochs': MAX_EPOCHS,
    'patience': PATIENCE,
    'min_delta': MIN_DELTA,
    'grad_clip': GRAD_CLIP,
    'use_amp': USE_AMP,
    'train_stride': TRAIN_STRIDE,
    'score_stride': SCORE_STRIDE,
    'lstm_dt_threshold_sec': LSTM_DT_THRESHOLD,
    'train_windows_per_group': TRAIN_WINDOWS_PER_GROUP,
    'val_windows_per_group': VAL_WINDOWS_PER_GROUP,
    'run_seeds': RUN_SEEDS,
}
with open(LSTM_CONFIG_PATH, 'w') as f:
    json.dump(lstm_config, f, indent=2)
print(f'\nSaved: {LSTM_CONFIG_PATH}')


=== Training LSTM-AE (seed=1321) ===
  Model parameters: 122,716
  epoch  1/40: train_loss=0.307595, val_loss=0.192481, time=22s ← BEST
  epoch  2/40: train_loss=0.143725, val_loss=0.097202, time=20s ← BEST
  epoch  3/40: train_loss=0.088281, val_loss=0.070200, time=21s ← BEST
  epoch  4/40: train_loss=0.066971, val_loss=0.055246, time=20s ← BEST
  epoch  5/40: train_loss=0.056727, val_loss=0.059466, time=20s
  epoch  6/40: train_loss=0.051886, val_loss=0.045911, time=21s ← BEST
  epoch  7/40: train_loss=0.048973, val_loss=0.044046, time=20s ← BEST
  epoch  8/40: train_loss=0.047096, val_loss=0.042148, time=20s ← BEST
  epoch  9/40: train_loss=0.045180, val_loss=0.040509, time=21s ← BEST
  epoch 10/40: train_loss=0.043679, val_loss=0.039933, time=20s ← BEST
  epoch 11/40: train_loss=0.042805, val_loss=0.038273, time=20s ← BEST
  epoch 12/40: train_loss=0.041685, val_loss=0.037770, time=20s ← BEST
  epoch 13/40: train_loss=0.041140, val_loss=0.037018, time=20s ← BEST
  epoch 14/40: tra

## 12. Score calval+test windows (TWO sets of aggregators)

**Aggregator pair A (general):** updated by ALL score-eligible windows.
Used for output point scores (lstm_score_max/mean).

**Aggregator pair B (clean-calibration-only):** updated ONLY by windows
where `split == 'calibration_val' AND is_clean == True`. Used to build
ECDF reference в Pass 2 calibration. This removes contamination where
clean point получает score из окна с DQ-tainted соседями.

Memory cost: 3 extra float32/uint16 arrays × 149M ≈ 1.5 GB.

In [12]:
print('Scoring calval + test windows...')
t0 = time.time()

model_main = LSTMAutoencoder(
    n_features=N_FEATURES, seq_len=SEQ_LEN, hidden_size=HIDDEN_SIZE,
    latent_dim=LATENT_DIM, num_layers=NUM_LAYERS, dropout=DROPOUT,
).to(device)
model_main.load_state_dict(torch.load(LSTM_MAIN_PATH, map_location=device))
model_main.eval()
print(f'Loaded main model from {LSTM_MAIN_PATH}')

model_alt = LSTMAutoencoder(
    n_features=N_FEATURES, seq_len=SEQ_LEN, hidden_size=HIDDEN_SIZE,
    latent_dim=LATENT_DIM, num_layers=NUM_LAYERS, dropout=DROPOUT,
).to(device)
model_alt.load_state_dict(torch.load(LSTM_ALT_PATH, map_location=device))
model_alt.eval()
print(f'Loaded alt model from {LSTM_ALT_PATH}')

total_rows_anno = pq.ParquetFile(INPUT_PATH).metadata.num_rows
print(f'Annotated_v3 total rows: {total_rows_anno:,}')
print('Allocating per-point aggregators...')

# General aggregators (all score-eligible windows)
lstm_max_main = np.full(total_rows_anno, -np.inf, dtype=np.float32)
lstm_sum_main = np.zeros(total_rows_anno, dtype=np.float32)
lstm_max_alt = np.full(total_rows_anno, -np.inf, dtype=np.float32)
lstm_sum_alt = np.zeros(total_rows_anno, dtype=np.float32)
lstm_count = np.zeros(total_rows_anno, dtype=np.uint16)

# Clean-calibration aggregators (only clean calval windows)
lstm_max_main_clean_cal = np.full(total_rows_anno, -np.inf, dtype=np.float32)
lstm_sum_main_clean_cal = np.zeros(total_rows_anno, dtype=np.float32)
lstm_count_clean_cal = np.zeros(total_rows_anno, dtype=np.uint16)

agg_mem = (lstm_max_main.nbytes + lstm_sum_main.nbytes + lstm_max_alt.nbytes
           + lstm_sum_alt.nbytes + lstm_count.nbytes
           + lstm_max_main_clean_cal.nbytes + lstm_sum_main_clean_cal.nbytes
           + lstm_count_clean_cal.nbytes)
print(f'Aggregator memory: {agg_mem / 1e9:.2f} GB')

score_meta_chunks = sorted(glob.glob(os.path.join(LSTM_SCORE_WINDOWS_DIR, 'meta_*.parquet')))
n_chunks_total = len(score_meta_chunks)
print(f'Found {n_chunks_total} score chunks')

Scoring calval + test windows...
Loaded main model from /content/drive/MyDrive/thesis_processed/models_v3_artifacts/lstm_main.pt
Loaded alt model from /content/drive/MyDrive/thesis_processed/models_v3_artifacts/lstm_alt.pt
Annotated_v3 total rows: 149,129,454
Allocating per-point aggregators...
Aggregator memory: 4.18 GB
Found 31 score chunks


In [13]:
@torch.no_grad()
def score_chunk(model, X_batch):
    """Score (batch, 60, 12) → per-timestep error (batch, 60)."""
    if USE_AMP:
        if NEW_AMP_API:
            with autocast(device_type='cuda'):
                recon = model(X_batch)
                err = ((recon - X_batch) ** 2).mean(dim=2)
        else:
            with autocast():
                recon = model(X_batch)
                err = ((recon - X_batch) ** 2).mean(dim=2)
    else:
        recon = model(X_batch)
        err = ((recon - X_batch) ** 2).mean(dim=2)
    return err.float().cpu().numpy()


for ci, mp in enumerate(score_meta_chunks):
    chunk_idx = int(os.path.basename(mp).split('_')[1].split('.')[0])
    win_path = os.path.join(LSTM_SCORE_WINDOWS_DIR, f'win_{chunk_idx:03d}.npz')
    if not os.path.exists(win_path):
        continue

    meta = pd.read_parquet(mp)
    windows = np.load(win_path)['windows']
    n_windows = len(windows)
    if n_windows == 0:
        continue

    start_rids = meta['start_row_id'].to_numpy()
    is_clean_arr = meta['is_clean'].to_numpy()
    split_arr = meta['split'].to_numpy()

    bs = BATCH_SIZE_SCORE
    errs_main = np.empty((n_windows, SEQ_LEN), dtype=np.float32)
    errs_alt = np.empty((n_windows, SEQ_LEN), dtype=np.float32)
    for bstart in range(0, n_windows, bs):
        bend = min(bstart + bs, n_windows)
        X_batch = torch.from_numpy(windows[bstart:bend]).to(device, non_blocking=True)
        errs_main[bstart:bend] = score_chunk(model_main, X_batch)
        errs_alt[bstart:bend] = score_chunk(model_alt, X_batch)

    # Update per-point aggregators
    for wi in range(n_windows):
        rid_start = start_rids[wi]
        rid_end = rid_start + SEQ_LEN
        em = errs_main[wi]
        ea = errs_alt[wi]

        # General aggregators (all windows)
        np.maximum(lstm_max_main[rid_start:rid_end], em, out=lstm_max_main[rid_start:rid_end])
        np.maximum(lstm_max_alt[rid_start:rid_end], ea, out=lstm_max_alt[rid_start:rid_end])
        lstm_sum_main[rid_start:rid_end] += em
        lstm_sum_alt[rid_start:rid_end] += ea
        lstm_count[rid_start:rid_end] += 1

        # Clean-calibration aggregators (только clean calval windows)
        if split_arr[wi] == 'calibration_val' and is_clean_arr[wi]:
            np.maximum(
                lstm_max_main_clean_cal[rid_start:rid_end], em,
                out=lstm_max_main_clean_cal[rid_start:rid_end]
            )
            lstm_sum_main_clean_cal[rid_start:rid_end] += em
            lstm_count_clean_cal[rid_start:rid_end] += 1

    print(f'  chunk {ci + 1}/{n_chunks_total}: {n_windows:,} windows scored', end='\r', flush=True)
    del meta, windows, errs_main, errs_alt
    gc.collect()

print()
print(f'\nScoring complete: {time.time() - t0:.0f}s')



Scoring complete: 160s


In [14]:
# Compute means
print('Computing per-point means from sums...')
with np.errstate(invalid='ignore', divide='ignore'):
    lstm_mean_main = np.where(lstm_count > 0,
                              lstm_sum_main / np.maximum(lstm_count, 1),
                              np.nan).astype(np.float32)
    lstm_mean_alt = np.where(lstm_count > 0,
                             lstm_sum_alt / np.maximum(lstm_count, 1),
                             np.nan).astype(np.float32)
    lstm_mean_main_clean_cal = np.where(
        lstm_count_clean_cal > 0,
        lstm_sum_main_clean_cal / np.maximum(lstm_count_clean_cal, 1),
        np.nan
    ).astype(np.float32)

lstm_max_main = np.where(lstm_count > 0, lstm_max_main, np.nan).astype(np.float32)
lstm_max_alt = np.where(lstm_count > 0, lstm_max_alt, np.nan).astype(np.float32)
lstm_max_main_clean_cal = np.where(
    lstm_count_clean_cal > 0, lstm_max_main_clean_cal, np.nan
).astype(np.float32)

del lstm_sum_main, lstm_sum_alt, lstm_sum_main_clean_cal
gc.collect()

n_covered = int((lstm_count > 0).sum())
n_clean_cal_covered = int((lstm_count_clean_cal > 0).sum())
print(f'Total points covered (general): {n_covered:,} / {total_rows_anno:,} '
      f'({n_covered / total_rows_anno * 100:.2f}%)')
print(f'Total points covered (clean-cal aggregator): {n_clean_cal_covered:,} '
      f'({n_clean_cal_covered / total_rows_anno * 100:.2f}%)')

del model_main, model_alt
if device.type == 'cuda':
    torch.cuda.empty_cache()

Computing per-point means from sums...
Total points covered (general): 62,038,980 / 149,129,454 (41.60%)
Total points covered (clean-cal aggregator): 20,560,005 (13.79%)


## 13. Build lstm_scores.parquet via 2 passes

**Pass 1:** collect clean-calibration ECDF references из clean-cal
aggregators (NOT general aggregators). Это key change для устранения
calibration contamination.

**Pass 2:** stream через annotated → compute percentile (vs clean-cal
ECDF) → write final parquet (with timestamp).

In [15]:
print('Pass 1: collecting clean-calibration ECDF references...')
t0 = time.time()

clean_calval_max = []
clean_calval_mean = []
clean_calval_max_phase = {p: [] for p in PHASE_NAMES}
clean_calval_mean_phase = {p: [] for p in PHASE_NAMES}

# Pass 1: stream annotated, collect clean-cal aggregator values for calval points
# Only points that have non-NaN clean-cal aggregator values qualify as reference
read_cols_2 = ['flight_id', 'flight_phase', DQ_HARD, DQ_SOFT, FEATURE_QUALITY]
pf2 = pq.ParquetFile(INPUT_PATH)

global_offset_2 = 0
for batch in pf2.iter_batches(batch_size=5_000_000, columns=read_cols_2):
    df = batch.to_pandas()
    n = len(df)
    rids = np.arange(global_offset_2, global_offset_2 + n, dtype=np.int64)
    global_offset_2 += n

    fid = df['flight_id'].to_numpy()
    is_calval = np.isin(fid, calibration_val_ids)

    if not is_calval.any():
        del df
        gc.collect()
        continue

    cal_rids = rids[is_calval]
    # Get clean-cal aggregator values
    max_clean_ref = lstm_max_main_clean_cal[cal_rids]
    mean_clean_ref = lstm_mean_main_clean_cal[cal_rids]

    # Reference is точки с finite clean-cal aggregator
    finite_ref = np.isfinite(max_clean_ref)
    if finite_ref.any():
        clean_calval_max.append(max_clean_ref[finite_ref])
        clean_calval_mean.append(mean_clean_ref[finite_ref])
        # Per phase
        phases_ref = df.loc[is_calval, 'flight_phase'].to_numpy()[finite_ref]
        for p in np.unique(phases_ref):
            p_mask = phases_ref == p
            clean_calval_max_phase[int(p)].append(max_clean_ref[finite_ref][p_mask])
            clean_calval_mean_phase[int(p)].append(mean_clean_ref[finite_ref][p_mask])

    del df
    gc.collect()

# Assert non-empty references (defensive)
assert len(clean_calval_max) > 0, 'No clean calibration LSTM max scores collected!'
assert len(clean_calval_mean) > 0, 'No clean calibration LSTM mean scores collected!'

ref_max_sorted = np.sort(np.concatenate(clean_calval_max))
ref_mean_sorted = np.sort(np.concatenate(clean_calval_mean))

assert len(ref_max_sorted) > 0, 'Empty ref_max_sorted!'
assert len(ref_mean_sorted) > 0, 'Empty ref_mean_sorted!'

print(f'Clean-cal aggregator reference points: {len(ref_max_sorted):,}')
print(f'  max range: [{ref_max_sorted[0]:.6f}, {ref_max_sorted[-1]:.6f}]')
print(f'  max P50/P90/P99/P99.9: '
      f'{np.quantile(ref_max_sorted, [0.5, 0.9, 0.99, 0.999]).round(6)}')
print(f'  mean range: [{ref_mean_sorted[0]:.6f}, {ref_mean_sorted[-1]:.6f}]')

ref_max_phase_sorted = {}
ref_mean_phase_sorted = {}
for p in PHASE_NAMES:
    if clean_calval_max_phase[p]:
        arr_max = np.concatenate(clean_calval_max_phase[p])
        arr_mean = np.concatenate(clean_calval_mean_phase[p])
        if len(arr_max) >= 100:
            ref_max_phase_sorted[p] = np.sort(arr_max)
            ref_mean_phase_sorted[p] = np.sort(arr_mean)
            print(f'  phase {PHASE_NAMES[p]:>10s}: ref n={len(arr_max):,}')
        else:
            print(f'  WARN phase {PHASE_NAMES[p]}: only {len(arr_max)} clean-cal points '
                  f'(< 100, will fall back to global)')

del clean_calval_max, clean_calval_mean, clean_calval_max_phase, clean_calval_mean_phase
del lstm_max_main_clean_cal, lstm_mean_main_clean_cal, lstm_count_clean_cal
gc.collect()
print(f'Pass 1 complete: {time.time() - t0:.0f}s')

Pass 1: collecting clean-calibration ECDF references...
Clean-cal aggregator reference points: 20,560,005
  max range: [0.000059, 26.985043]
  max P50/P90/P99/P99.9: [0.008212 0.067871 0.483075 1.715527]
  mean range: [0.000059, 26.985043]
  phase     ground: ref n=16,604
  phase    takeoff: ref n=33,441
  phase      climb: ref n=3,832,976
  phase     cruise: ref n=10,799,389
  phase    descent: ref n=3,476,641
  phase   approach: ref n=2,236,525
  phase    landing: ref n=164,429
Pass 1 complete: 11s


In [16]:
# Pass 2: build output parquet with timestamp
print('\nPass 2: writing lstm_scores.parquet (with timestamp)...')
t0 = time.time()

read_cols_3 = ['flight_id', 'timestamp', 'flight_phase',
               DQ_HARD, DQ_SOFT, FEATURE_QUALITY]
pf3 = pq.ParquetFile(INPUT_PATH)
writer = None
total_written = 0
n_phase_fallback = 0
global_offset_3 = 0

for batch in pf3.iter_batches(batch_size=5_000_000, columns=read_cols_3):
    df = batch.to_pandas()
    n = len(df)
    rids = np.arange(global_offset_3, global_offset_3 + n, dtype=np.int64)
    global_offset_3 += n

    fid = df['flight_id'].to_numpy()
    is_calval = np.isin(fid, calibration_val_ids)
    is_test = np.isin(fid, test_ids)
    is_eligible = is_calval | is_test

    if not is_eligible.any():
        del df
        gc.collect()
        continue

    elig_rids = rids[is_eligible]
    max_m = lstm_max_main[elig_rids]
    mean_m = lstm_mean_main[elig_rids]
    max_a = lstm_max_alt[elig_rids]
    mean_a = lstm_mean_alt[elig_rids]
    count = lstm_count[elig_rids].astype(np.int32)
    phases = df.loc[is_eligible, 'flight_phase'].to_numpy().astype('int8')

    finite_max = np.isfinite(max_m)
    max_global_pct = np.full(len(elig_rids), np.nan, dtype=np.float32)
    mean_global_pct = np.full(len(elig_rids), np.nan, dtype=np.float32)
    if finite_max.any():
        max_global_pct[finite_max] = (
            np.searchsorted(ref_max_sorted, max_m[finite_max], side='right')
            / len(ref_max_sorted)
        )
        mean_global_pct[finite_max] = (
            np.searchsorted(ref_mean_sorted, mean_m[finite_max], side='right')
            / len(ref_mean_sorted)
        )

    max_phase_pct = np.full(len(elig_rids), np.nan, dtype=np.float32)
    mean_phase_pct = np.full(len(elig_rids), np.nan, dtype=np.float32)
    for p, ref_p_max in ref_max_phase_sorted.items():
        p_mask = (phases == p) & finite_max
        if not p_mask.any():
            continue
        max_phase_pct[p_mask] = (
            np.searchsorted(ref_p_max, max_m[p_mask], side='right')
            / len(ref_p_max)
        )
        ref_p_mean = ref_mean_phase_sorted[p]
        mean_phase_pct[p_mask] = (
            np.searchsorted(ref_p_mean, mean_m[p_mask], side='right')
            / len(ref_p_mean)
        )

    fb_mask_max = np.isnan(max_phase_pct) & np.isfinite(max_global_pct)
    n_phase_fallback += int(fb_mask_max.sum())
    max_phase_pct[fb_mask_max] = max_global_pct[fb_mask_max]
    fb_mask_mean = np.isnan(mean_phase_pct) & np.isfinite(mean_global_pct)
    mean_phase_pct[fb_mask_mean] = mean_global_pct[fb_mask_mean]

    dq_hard_arr = df.loc[is_eligible, DQ_HARD].to_numpy().astype('uint8')
    dq_soft_arr = df.loc[is_eligible, DQ_SOFT].to_numpy().astype('uint8')
    fq_arr = df.loc[is_eligible, FEATURE_QUALITY].to_numpy().astype('uint8')
    is_clean_ref = ((dq_hard_arr == 0) & (dq_soft_arr == 0) & (fq_arr == 0)).astype('uint8')

    split_arr_out = np.where(is_calval[is_eligible], 'calibration_val', 'test')

    out_df = pd.DataFrame({
        'row_id': elig_rids,
        'flight_id': fid[is_eligible].astype('int64'),
        'timestamp': df.loc[is_eligible, 'timestamp'].to_numpy(),
        'flight_phase': phases,
        'split': pd.Categorical(split_arr_out, categories=['calibration_val', 'test']),
        DQ_HARD: dq_hard_arr,
        DQ_SOFT: dq_soft_arr,
        FEATURE_QUALITY: fq_arr,
        'is_clean_reference': is_clean_ref,
        'lstm_score_max_raw': max_m,
        'lstm_score_mean_raw': mean_m,
        'lstm_score_count': count,
        'lstm_score_max_alt_raw': max_a,
        'lstm_score_mean_alt_raw': mean_a,
        'lstm_score_max_global_pct': max_global_pct,
        'lstm_score_max_phase_pct': max_phase_pct,
        'lstm_score_mean_global_pct': mean_global_pct,
        'lstm_score_mean_phase_pct': mean_phase_pct,
    })

    table = pa.Table.from_pandas(out_df, preserve_index=False)
    if writer is None:
        writer = pq.ParquetWriter(LSTM_SCORES_PATH, table.schema, compression='snappy')
    writer.write_table(table)
    total_written += len(out_df)

    print(f'  scored so far: {total_written:,}', end='\r', flush=True)

    del df, out_df, table
    gc.collect()

if writer is not None:
    writer.close()

print()
print(f'\nPass 2 complete: {time.time() - t0:.0f}s')
print(f'Total written: {total_written:,}')
print(f'Phase pct fallback to global: {n_phase_fallback:,}')
print(f'File size: {os.path.getsize(LSTM_SCORES_PATH) / 1e6:.1f} MB')


Pass 2: writing lstm_scores.parquet (with timestamp)...


Pass 2 complete: 386s
Total written: 63,019,803
Phase pct fallback to global: 0
File size: 2712.2 MB


## 14. Stability analysis (main vs alt)

In [17]:
print('Stability analysis: main vs alt on calval...')

calval_data = pq.read_table(
    LSTM_SCORES_PATH,
    columns=['split', 'is_clean_reference',
             'lstm_score_max_raw', 'lstm_score_max_alt_raw'],
    filters=[('split', '=', 'calibration_val')]
).to_pandas()

calval_data = calval_data.dropna(subset=['lstm_score_max_raw', 'lstm_score_max_alt_raw'])
n_all = len(calval_data)
n_clean = int((calval_data['is_clean_reference'] == 1).sum())

print(f'Calval with both scores: {n_all:,}')
print(f'  Clean subset: {n_clean:,} ({n_clean / n_all * 100:.2f}%)')


def jaccard_top_pct(scores_a, scores_b, top_pct=0.01):
    n = len(scores_a)
    n_top = max(1, int(n * top_pct))
    if n_top >= n:
        return 1.0
    top_a = set(np.argpartition(scores_a, -n_top)[-n_top:])
    top_b = set(np.argpartition(scores_b, -n_top)[-n_top:])
    inter = len(top_a & top_b)
    union = len(top_a | top_b)
    return inter / union if union > 0 else 0.0


def compute_stability(s_main, s_alt):
    return {
        'n_points': len(s_main),
        'jaccard_top_1pct': float(jaccard_top_pct(s_main, s_alt, 0.01)),
        'jaccard_top_0_5pct': float(jaccard_top_pct(s_main, s_alt, 0.005)),
        'jaccard_top_0_1pct': float(jaccard_top_pct(s_main, s_alt, 0.001)),
        'pearson': float(np.corrcoef(s_main, s_alt)[0, 1]),
    }


s_main_all = calval_data['lstm_score_max_raw'].to_numpy()
s_alt_all = calval_data['lstm_score_max_alt_raw'].to_numpy()
stab_all = compute_stability(s_main_all, s_alt_all)

clean = calval_data[calval_data['is_clean_reference'] == 1]
s_main_c = clean['lstm_score_max_raw'].to_numpy()
s_alt_c = clean['lstm_score_max_alt_raw'].to_numpy()
stab_clean = compute_stability(s_main_c, s_alt_c)

stability = {
    'level': 'point (lstm_score_max_raw)',
    'all_calibration_val': stab_all,
    'clean_calibration_val': stab_clean,
}

print('\n=== Stability ALL ===')
for k, v in stab_all.items():
    print(f'  {k}: {v}')
print('\n=== Stability CLEAN ===')
for k, v in stab_clean.items():
    print(f'  {k}: {v}')

if stab_clean['jaccard_top_1pct'] > 0.6:
    print('\nClean stability: STABLE')
elif stab_clean['jaccard_top_1pct'] > 0.4:
    print('\nClean stability: MODERATE')
else:
    print('\nClean stability: WEAK')

with open(LSTM_STABILITY_PATH, 'w') as f:
    json.dump(stability, f, indent=2)
print(f'Saved: {LSTM_STABILITY_PATH}')

del calval_data, clean
gc.collect()

Stability analysis: main vs alt on calval...
Calval with both scores: 21,212,220
  Clean subset: 20,919,065 (98.62%)

=== Stability ALL ===
  n_points: 21212220
  jaccard_top_1pct: 0.5654009217270021
  jaccard_top_0_5pct: 0.5864926517332935
  jaccard_top_0_1pct: 0.5761628770991232
  pearson: 0.898251881501101

=== Stability CLEAN ===
  n_points: 20919065
  jaccard_top_1pct: 0.5239975521622566
  jaccard_top_0_5pct: 0.5292113804497207
  jaccard_top_0_1pct: 0.4812533191715348
  pearson: 0.8708270636014567

Clean stability: MODERATE
Saved: /content/drive/MyDrive/thesis_processed/models_v3_artifacts/lstm_stability.json


33

## 15. Top-10 anomaly preview (streaming top-k) — with dq_soft in tainted

In [18]:
print('\nTop-10 anomaly preview...')
N_TOP = 10
top_overall = pd.DataFrame()
top_clean = pd.DataFrame()
top_tainted = pd.DataFrame()

pf_scores = pq.ParquetFile(LSTM_SCORES_PATH)
for batch in pf_scores.iter_batches(
        batch_size=2_000_000,
        columns=['flight_id', 'split', 'flight_phase',
                 'lstm_score_max_raw', 'lstm_score_max_global_pct',
                 'lstm_score_max_phase_pct', 'lstm_score_count',
                 DQ_HARD, DQ_SOFT, FEATURE_QUALITY, 'is_clean_reference']):
    df = batch.to_pandas().dropna(subset=['lstm_score_max_raw'])
    if len(df) == 0:
        continue

    candidates = pd.concat([top_overall, df]) if len(top_overall) else df
    top_overall = candidates.nlargest(N_TOP, 'lstm_score_max_raw')

    df_clean = df[df['is_clean_reference'] == 1]
    if len(df_clean) > 0:
        candidates_c = pd.concat([top_clean, df_clean]) if len(top_clean) else df_clean
        top_clean = candidates_c.nlargest(N_TOP, 'lstm_score_max_raw')

    # DQ-tainted: now includes dq_soft_flag
    df_tainted = df[
        (df[DQ_HARD] == 1) | (df[DQ_SOFT] == 1) | (df[FEATURE_QUALITY] == 1)
    ]
    if len(df_tainted) > 0:
        candidates_t = pd.concat([top_tainted, df_tainted]) if len(top_tainted) else df_tainted
        top_tainted = candidates_t.nlargest(N_TOP, 'lstm_score_max_raw')


def _print_top(df_top, title, with_flags=False):
    print(f'\n--- {title} ---')
    if len(df_top) == 0:
        print('  (no points)')
        return
    df_top = df_top.copy()
    df_top['phase_name'] = df_top['flight_phase'].map(
        lambda p: PHASE_NAMES.get(p, 'unknown'))
    cols = ['flight_id', 'split', 'phase_name', 'lstm_score_max_raw',
            'lstm_score_max_global_pct', 'lstm_score_max_phase_pct',
            'lstm_score_count']
    if with_flags:
        cols += [DQ_HARD, DQ_SOFT, FEATURE_QUALITY]
    print(df_top[cols].to_string(index=False))


_print_top(top_overall, 'Top-10 overall by lstm_score_max_raw', with_flags=True)
_print_top(top_clean, 'Top-10 CLEAN (potential operational anomalies)')
_print_top(top_tainted, 'Top-10 DQ-TAINTED (likely artifacts)', with_flags=True)

del top_overall, top_clean, top_tainted
gc.collect()


Top-10 anomaly preview...

--- Top-10 overall by lstm_score_max_raw ---
 flight_id           split phase_name  lstm_score_max_raw  lstm_score_max_global_pct  lstm_score_max_phase_pct  lstm_score_count  dq_hard_flag  dq_soft_flag  feature_quality_flag
 256851839 calibration_val     cruise           48.285442                        1.0                       1.0                 4             0             1                     0
 256942197            test      climb           39.474438                        1.0                       1.0                 4             0             1                     0
 256917380 calibration_val      climb           39.211037                        1.0                       1.0                 4             0             1                     0
 256936974            test   approach           34.615677                        1.0                       1.0                 4             0             1                     0
 256942197            test      

0

## 16. Cleanup + summary

In [19]:
print('\nCleaning up temp dirs...')
for tmp_dir in [LSTM_TRAIN_WINDOWS_DIR, LSTM_SCORE_WINDOWS_DIR]:
    files = glob.glob(os.path.join(tmp_dir, '*'))
    for f in files:
        os.remove(f)
    try:
        os.rmdir(tmp_dir)
    except OSError:
        pass

del lstm_max_main, lstm_max_alt, lstm_mean_main, lstm_mean_alt, lstm_count
gc.collect()

print('\n' + '=' * 60)
print('03.4 SUMMARY')
print('=' * 60)
print(f'\nLSTM-AE runs: {list(RUN_SEEDS.keys())}')
for r, info in run_results.items():
    print(f'  {r}: best_val_loss={info["best_val_loss"]:.6f} '
          f'@ epoch {info["best_epoch"]}')

print(f'\nScoring coverage: {n_covered:,} / {total_rows_anno:,} '
      f'({n_covered / total_rows_anno * 100:.2f}%)')
print(f'Clean-cal aggregator coverage: {n_clean_cal_covered:,} '
      f'({n_clean_cal_covered / total_rows_anno * 100:.2f}%)')
print(f'Phase fallback: {n_phase_fallback:,}')
print(f'\nStability (clean Jaccard top-1%): {stab_clean["jaccard_top_1pct"]:.4f}')
print(f'Stability (clean Pearson): {stab_clean["pearson"]:.4f}')

print(f'\nArtifacts:')
for f in [LSTM_MAIN_PATH, LSTM_ALT_PATH, LSTM_CONFIG_PATH,
          LSTM_HIST_MAIN_PATH, LSTM_HIST_ALT_PATH,
          LSTM_SCORES_PATH, LSTM_STABILITY_PATH]:
    if os.path.exists(f):
        size_mb = os.path.getsize(f) / 1e6
        print(f'  ✓ {os.path.basename(f):<35s} ({size_mb:>7.2f} MB)')

print('\n03.4 complete. Ready for 03.5 (ensemble fusion + events + evaluation).')


Cleaning up temp dirs...

03.4 SUMMARY

LSTM-AE runs: ['main', 'alt']
  main: best_val_loss=0.030269 @ epoch 40
  alt: best_val_loss=0.032215 @ epoch 33

Scoring coverage: 62,038,980 / 149,129,454 (41.60%)
Clean-cal aggregator coverage: 20,560,005 (13.79%)
Phase fallback: 0

Stability (clean Jaccard top-1%): 0.5240
Stability (clean Pearson): 0.8708

Artifacts:
  ✓ lstm_main.pt                        (   0.50 MB)
  ✓ lstm_alt.pt                         (   0.50 MB)
  ✓ lstm_config.json                    (   0.00 MB)
  ✓ lstm_training_history_main.json     (   0.00 MB)
  ✓ lstm_training_history_alt.json      (   0.00 MB)
  ✓ lstm_scores.parquet                 (2712.17 MB)
  ✓ lstm_stability.json                 (   0.00 MB)

03.4 complete. Ready for 03.5 (ensemble fusion + events + evaluation).
